# 002_images_completed.ipynb — Amazon Bedrock

Configura IMAGE_PATH con una imagen real, o añade images/prop3.png en esta subcarpeta. Converse recibe sus bytes.

Ejecuta las celdas en orden con credenciales AWS y acceso al modelo. La configuración usa servicios reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Amazon Bedrock: credenciales AWS del entorno o perfil.
import os
from pathlib import Path

import boto3
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "BEDROCK" / "README.md").is_file()
)
bedrock_root = repo_root / "BEDROCK"
load_dotenv(bedrock_root / ".env", override=False)
exercise_dir = bedrock_root / "FeatsOfClaude"
os.chdir(exercise_dir)

session = boto3.Session(
    profile_name=os.getenv("AWS_PROFILE"),
    region_name=os.getenv("AWS_REGION", "us-west-2"),
)
client = session.client("bedrock-runtime")
model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-6")


In [ ]:
# Mensajes y respuestas en el formato nativo de Converse.
def message_content(content):
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
        return [{"text": content}]
    if not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita una lista de bloques de contenido.")
    for block in content:
        if "text" in block and not block["text"].strip():
            raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, content):
    messages.append({"role": "user", "content": message_content(content)})


def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": message_content(content)})


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice="auto",
    max_tokens=4000,
):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens},
    }
    if stop_sequences:
        params["inferenceConfig"]["stopSequences"] = list(stop_sequences)
    if system:
        params["system"] = [{"text": system}]
    if tools:
        choices = {"auto": {"auto": {}}, "any": {"any": {}}}
        choice = choices.get(tool_choice, {"tool": {"name": tool_choice}})
        selected_tools = list(tools)
        params["toolConfig"] = {"tools": selected_tools, "toolChoice": choice}
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]
    text = "\n".join(part["text"] for part in parts if "text" in part)
    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": text,
        "usage": response["usage"],
    }


In [ ]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification. Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree density analysis. Examine the overall distribution and density of trees on the property:
   - Identify areas where trees are closely packed together
   - Assess the spacing between tree canopies (isolated, clustered, or continuous)
   - Evaluate whether fire could easily spread from tree to tree based on proximity
   - Note any particularly dense wooded areas in relation to the main residence

3. Access assessment. Evaluate emergency vehicle access to the primary residence:
   - Identify clear paths, roads, or driveways leading to or from the main structure
   - Assess whether access routes are wide enough for emergency vehicles
   - Note any potential obstructions (gates, narrow passages, overhanging vegetation)
   - Consider if multiple access points exist or if there is only a single entry/exit route

4. Roof overhang analysis. Examine tree branches in relation to the main residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang or contact points between branches and structure
   - Assess proximity of branches to chimneys, vents, or other roof openings if visible

5. Fire risk rating. Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): Well-spaced trees with no canopy continuity, clear access routes, no branches overhanging the roof
   - Rating 2 (Moderate Risk): Some clustered trees but with breaks in continuity, adequate access routes, minimal roof overhang (<25%)
   - Rating 3 (High Risk): Densely packed trees with connected canopies, limited or narrow access routes, significant roof overhang (25-50%)
   - Rating 4 (Severe Risk): Continuous tree canopy coverage across property, restricted or obstructed access routes, extensive roof overhang (>50%)

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [ ]:
image_path = Path(os.getenv("IMAGE_PATH", str(exercise_dir / "images" / "prop3.png"))).expanduser()
if not image_path.is_file():
    raise FileNotFoundError("Configura IMAGE_PATH en BEDROCK/.env o añade FeatsOfClaude/images/prop3.png.")
formats = {".png": "png", ".jpg": "jpeg", ".jpeg": "jpeg", ".gif": "gif", ".webp": "webp"}
image_format = formats.get(image_path.suffix.lower())
if not image_format:
    raise ValueError("La imagen debe ser PNG, JPEG, GIF o WebP.")
image_bytes = image_path.read_bytes()
messages = []
add_user_message(messages, [
    {"image": {"format": image_format, "source": {"bytes": image_bytes}}},
    {"text": prompt},
])
response = chat(messages)
response["text"]
